In [1]:
# req libs
from rerankers import mmr, steck_2018_re_rank
import numpy as np
from metrics import ild, miscalibration


### Task 3.1 - Rerankers

Every reranker in [rerankers.py](rerankers.py) has the same interface:

```
rerank(scores_u, candidates_u, k, lam, **context) -> k item ids, best first
```

* `scores_u`: the base model's scores for one user, indexed by item id (one row of a score matrix).
* `candidates_u`: the item ids the reranker may choose from (the user's top-N pool).
* `lam` in [0, 1]: 0 returns the base ranking, 1 puts all weight on the beyond-accuracy objective.
* `context`: what the method needs, e.g. `genres` for MMR.

The sanity checks below run on the toy catalogue from [task_2_1_beyond_accuracy.ipynb](task_2_1_beyond_accuracy.ipynb).

In [ ]:
# Toy catalogue from task_2_1_beyond_accuracy: 6 items over genres A, B, C
G = np.array([[1, 0, 0], [1, 0, 0], [0, 1, 0], [1, 1, 0], [0, 0, 1], [1, 1, 1]])
scores = np.array([0.9, 0.8, 0.3, 0.2, 0.5, 0.1])   # base ranking: 0 > 1 > 4 > 2 > 3 > 5
cands = np.arange(6)
subset = np.array([1, 2, 4, 5])                      # a pool that leaves out the top item
LAMBDAS = [0, 0.25, 0.5, 0.75, 1]


def check_basics(rerank, k=3, **context):
    """Checks every reranker must pass: length, no duplicates, only candidates, deterministic,
    and λ = 0 returns the base ranking of the pool."""
    for pool in (cands, subset):
        for lam in LAMBDAS:
            out = list(rerank(scores, pool, k=k, lam=lam, **context))
            assert len(out) == k and len(set(out)) == k, f"λ={lam}: wrong length or duplicates {out}"
            assert set(out) <= set(pool.tolist()), f"λ={lam}: returned a non-candidate {out}"
            assert out == list(rerank(scores, pool, k=k, lam=lam, **context)), f"λ={lam}: not deterministic"
        base = pool[np.argsort(-scores[pool], kind="stable")][:k].tolist()
        assert list(rerank(scores, pool, k=k, lam=0, **context)) == base, "λ = 0 changed the base ranking"

    try:
        rerank(scores, cands[:2], k=3, lam=0.5, **context)
    except ValueError:
        pass
    else:
        raise AssertionError("k larger than the pool should raise ValueError")

In [ ]:
# --- MMR
check_basics(mmr, genres=G)

# λ = 0.9, k = 2: item 0 (A) is picked first. Item 1 is also A (distance 0), while 2 (B) and 4 (C) are at
# distance 1; the small relevance weight breaks that tie towards the higher-scored item 4.
assert list(mmr(scores, cands, k=2, lam=0.9, genres=G)) == [0, 4]

# k = 3: item 2 (B) is at distance 1 from both 0 and 4, so the list is three disjoint genres
assert list(mmr(scores, cands, k=3, lam=0.9, genres=G)) == [0, 4, 2]

# measured with the report's metric: ILD goes from 2/3 for the base list [0, 1, 4] to 1
assert np.isclose(ild([mmr(scores, cands, k=3, lam=0, genres=G)], G)[0], 2 / 3)
assert np.isclose(ild([mmr(scores, cands, k=3, lam=0.9, genres=G)], G)[0], 1)
print("MMR toy checks OK")